# Snowball pricing and hedging walkthrough

This notebook values a discrete-barrier Snowball from the investor's perspective, then hedges the corresponding short claim. Run it in an environment with the package and its notebook extras installed. The notebook uses smaller experiments than the full research script.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from snowball import Market, PDEGrid, Snowball, price_monte_carlo, solve_decomposition, solve_pde
from snowball.greeks import PDERisk
from snowball.hedging import delta_hedge
from snowball.simulation import simulate_paths

terms, market = Snowball(), Market()
grid = PDEGrid(space_steps=640)
pd.DataFrame({
    "KI observations": [len(terms.schedule.ki_steps)],
    "KO observations": [len(terms.schedule.ko_steps)],
    "First KO time (years)": [terms.schedule.times[terms.schedule.ko_steps[0]]],
    "Maturity (years)": [terms.maturity],
})

## Two pricing engines

Each price is per unit notional. The net claim settles the coupon or the realized loss. Gross redemption includes the principal component payable at the same stopping time. A deposit of par at issue gives the investor gross PV less par.

In [ ]:
mc = price_monte_carlo(terms, market, n_paths=60_000, seed=42)
net = solve_pde(terms, market, grid)
gross = solve_pde(terms, market, grid, cashflow="gross")
legs = solve_decomposition(terms, market, grid)

pd.DataFrame([{
    "MC net PV": mc.price,
    "MC 95% lower": mc.estimate.confidence_interval[0],
    "MC 95% upper": mc.estimate.confidence_interval[1],
    "PDE net PV": net.price,
    "Four-leg net PV": legs.total,
    "PDE gross PV less par": gross.price - terms.notional,
}])

## Conditional Delta

The same spot can produce different exposures after KI. The barrier and reference levels stay fixed when bumping the current spot. The date slice is the close one trading day before the first monthly KO check.

In [ ]:
risk = PDERisk(net)
spots = np.linspace(0.77, 1.13, 145)
fig, ax = plt.subplots(figsize=(8, 3.4), layout="constrained")
for knocked_in, label in ((False, "Clean"), (True, "Previously knocked in")):
    profile = risk.evaluate(spots, step=20, knocked_in=knocked_in)
    ax.plot(spots, profile.delta, label=label)
for barrier in (terms.ki_barrier, terms.ko_barrier):
    ax.axvline(barrier, color="0.5", linestyle="--", linewidth=0.9)
ax.set(xlabel="Spot", ylabel="Investor Delta (shares per contract)")
ax.legend()
plt.show()

## Hedge comparison

The issuer sells the net claim at its PDE value. The hedge buys the investor Delta, finances inventory through the cash account and liquidates at settlement. The strategies use identical risk-neutral paths. P&L is expressed at a common maturity date.

In [ ]:
paths = simulate_paths(terms, market, 2_000, seed=7701)
rows = []
for name, interval, cost in [("Unhedged", None, 0), ("Weekly", 5, 0),
                             ("Daily", 1, 0), ("Daily", 1, 20)]:
    hedge = delta_hedge(net, paths, rebalance_every=interval, cost_bps=cost)
    rows.append({
        "Strategy": name,
        "Cost (bp)": cost,
        "Mean P&L (bp)": 1e4 * hedge.pnl_at_maturity.mean(),
        "P&L std. dev. (bp)": 1e4 * hedge.pnl_at_maturity.std(ddof=1),
        "Turnover (times notional)": hedge.turnover_pv.mean(),
    })
pd.DataFrame(rows).round(3)